# Databricks/PySpark dla Data Analyst/Engineer — Moduł 7: Funkcje okienkowe w Sparku

Funkcje okienkowe pozwalają liczyć wartości "w kontekście" grupy wierszy (np. ranking w
obrębie miasta, suma narastająca w czasie), BEZ zwijania wyniku do jednego wiersza na
grupę -- w przeciwieństwie do `groupBy`. To dokładnie ten sam koncept co w Module 11
kursu SQL, teraz przez `pyspark.sql.window.Window`.

## Spis treści
1. [Window: partitionBy i orderBy](#sec1)
2. [row_number, rank, dense_rank](#sec2)
3. [Sumy narastające (running totals)](#sec3)
4. [lag i lead](#sec4)
5. [Podsumowanie i ćwiczenia](#sec5)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("kurs_spark").getOrCreate()
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. Window: partitionBy i orderBy

Obiekt `Window` definiuje "okno" -- jak podzielić dane na grupy (`partitionBy`) i w jakiej
kolejności je przeglądać wewnątrz grupy (`orderBy`). Sam w sobie nic nie liczy -- dopiero
przekazujemy go do funkcji okienkowej przez `.over(okno)`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

okno_wg_miasta = Window.partitionBy("miasto").orderBy(F.col("dni_od_rejestracji").desc())

klienci.withColumn(
    "pozycja_w_miescie", F.row_number().over(okno_wg_miasta)
).orderBy("miasto", "pozycja_w_miescie").show(10)


> ⚡ **Window.partitionBy() to nie to samo co groupBy()**
>
> `groupBy()` ZWIJA wiele wierszy w jeden na grupę. `Window.partitionBy()` NIE zwija nic -- każdy oryginalny wiersz zostaje na swoim miejscu, tylko dostaje dodatkową kolumnę policzoną 'w kontekście' swojej grupy. To główny powód, dla którego funkcje okienkowe są tak przydatne: ranking, suma narastająca czy porównanie z poprzednim wierszem -- bez utraty szczegółowych danych.

<a id="sec2"></a>
## 2. row_number, rank, dense_rank

Trzy najczęstsze funkcje rankingujące, różniące się zachowaniem przy REMISACH (tych
samych wartościach w kolumnie `orderBy`):

- `row_number()` -- zawsze unikalny numer 1, 2, 3, ... (nawet przy remisie),
- `rank()` -- remisujące wiersze dostają ten sam numer, a następny numer PRZESKAKUJE (1, 1, 3),
- `dense_rank()` -- remisujące wiersze dostają ten sam numer, a następny numer NIE przeskakuje (1, 1, 2).

In [ ]:
okno_cenowe = Window.partitionBy("kategoria").orderBy(F.col("cena").desc())

produkty.withColumn("row_number", F.row_number().over(okno_cenowe)) \
    .withColumn("rank", F.rank().over(okno_cenowe)) \
    .withColumn("dense_rank", F.dense_rank().over(okno_cenowe)) \
    .orderBy("kategoria", "row_number").show(15)


> ⚠️ **Top-N per grupa -- klasyczny wzorzec z row_number()**
>
> Najczęstsze zastosowanie: 'znajdź 3 najdroższe produkty w każdej kategorii'. Liczysz `row_number()` w oknie posortowanym malejąco po cenie, a potem filtrujesz `WHERE row_number <= 3` -- coś, czego nie da się wyrazić samym `groupBy()`.

In [ ]:
produkty.withColumn("ranking", F.row_number().over(okno_cenowe)).filter(
    F.col("ranking") <= 3
).orderBy("kategoria", "ranking").show(20)


<a id="sec3"></a>
## 3. Sumy narastające (running totals)

Podając zakres wierszy do okna przez `.rowsBetween(...)`, możemy liczyć sumy/średnie
narastające -- np. `Window.unboundedPreceding` do bieżącego wiersza oznacza "wszystkie
wiersze od początku partycji aż do teraz".

In [ ]:
okno_narastajace = (
    Window.partitionBy("klient_id")
    .orderBy("data_zamowienia")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

zamowienia_z_wartoscia = zamowienia.join(pozycje_zamowien, "zamowienie_id").groupBy(
    "zamowienie_id", "klient_id", "data_zamowienia"
).agg(F.sum(F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat"))).alias("wartosc"))

zamowienia_z_wartoscia.withColumn(
    "suma_narastajaca", F.round(F.sum("wartosc").over(okno_narastajace), 2)
).orderBy("klient_id", "data_zamowienia").show(10)


> 🧱 **Domyślny zakres okna, gdy pominiesz rowsBetween**
>
> Jeśli podasz tylko `orderBy` bez `.rowsBetween(...)`, Spark domyślnie liczy sumę narastającą DOKŁADNIE tak jak wyżej (`unboundedPreceding` do `currentRow`) -- ale jawne podanie zakresu jest czytelniejsze i unika niespodzianek przy remisach w kolumnie `orderBy` (wiersze z tą samą wartością mogą się wtedy 'zlepić' w jedną grupę zakresu, tzw. `RANGE` zamiast `ROWS`).

<a id="sec4"></a>
## 4. lag i lead

`lag(kolumna, n)` zwraca wartość z wiersza `n` POZYCJI WCZEŚNIEJ w oknie, `lead(kolumna, n)`
-- `n` pozycji PÓŹNIEJ. Klasyczne zastosowanie: porównanie z poprzednim/następnym
zamówieniem tego samego klienta.

In [ ]:
okno_czasowe = Window.partitionBy("klient_id").orderBy("data_zamowienia")

zamowienia_z_wartoscia.withColumn(
    "poprzednia_wartosc", F.lag("wartosc", 1).over(okno_czasowe)
).withColumn(
    "zmiana", F.round(F.col("wartosc") - F.col("poprzednia_wartosc"), 2)
).orderBy("klient_id", "data_zamowienia").show(10)


> ⚠️ **Pierwszy/ostatni wiersz w partycji -- lag/lead zwraca NULL**
>
> Dla najwcześniejszego zamówienia klienta `lag(...)` nie ma skąd wziąć poprzedniej wartości -- zwróci `NULL`. To oczekiwane zachowanie, ale trzeba je obsłużyć (np. `fillna` albo świadomie zostawić jako brak danych), zwłaszcza przed dalszymi obliczeniami na kolumnie `zmiana`.

<a id="sec5"></a>
## 5. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- `Window.partitionBy().orderBy()` -- definiowanie okna bez zwijania wierszy (w przeciwieństwie do `groupBy`),
- `row_number()`, `rank()`, `dense_rank()` -- różnice w obsłudze remisów,
- wzorzec Top-N per grupa przez `row_number()` + filtr,
- sumy/średnie narastające przez `.rowsBetween(...)`,
- `lag()`/`lead()` -- dostęp do wartości z sąsiednich wierszy w oknie.

### 📝 Ćwiczenie 1: Ranking pracowników w regionie

Dla `pracownicy`, policz `row_number()` w oknie partycjonowanym po `region` i posortowanym po `pracownik_id`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql.window import Window
from pyspark.sql import functions as F

okno = Window.partitionBy("region").orderBy("pracownik_id")
pracownicy.withColumn("numer_w_regionie", F.row_number().over(okno)).orderBy(
    "region", "numer_w_regionie"
).show()
```

</details>

### 📝 Ćwiczenie 2: 2 najtańsze produkty w każdej kategorii

Użyj `row_number()` w oknie posortowanym ROSNĄCO po `cena` w obrębie `kategoria`, żeby znaleźć 2 najtańsze produkty w każdej kategorii.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
okno = Window.partitionBy("kategoria").orderBy(F.col("cena").asc())
produkty.withColumn("ranking", F.row_number().over(okno)).filter(
    F.col("ranking") <= 2
).orderBy("kategoria", "ranking").show()
```

</details>

### 📝 Ćwiczenie 3: Suma narastająca liczby zamówień w czasie

Policz liczbę zamówień per dzień (`data_zamowienia`), a następnie sumę narastającą tej liczby w czasie (jedna partycja -- całość, posortowana po dacie).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dzienne = zamowienia.groupBy("data_zamowienia").agg(F.count("*").alias("liczba"))

okno = Window.orderBy("data_zamowienia").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
dzienne.withColumn("suma_narastajaca", F.sum("liczba").over(okno)).orderBy(
    "data_zamowienia"
).show(10)
```

</details>

### 📝 Ćwiczenie 4: Porównanie z poprzednim zamówieniem

Dla każdego klienta, użyj `lag()`, żeby dodać kolumnę ze STATUSEM jego poprzedniego zamówienia (posortowane po `data_zamowienia`).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
okno = Window.partitionBy("klient_id").orderBy("data_zamowienia")
zamowienia.withColumn(
    "poprzedni_status", F.lag("status", 1).over(okno)
).orderBy("klient_id", "data_zamowienia").show(10)
```

</details>

> 🔥 **Wyzwanie: klienci z rosnącą wartością trzech ostatnich zamówień**
>
> Dla każdego klienta policz wartość każdego zamówienia (jak w sekcji 3), dodaj kolumny `wartosc_poprzedniego` i `wartosc_przedpoprzedniego` (`lag(1)` i `lag(2)`), a następnie znajdź klientów, u których ostatnie TRZY zamówienia mają ŚCIŚLE rosnącą wartość (`wartosc_przedpoprzedniego < wartosc_poprzedniego < wartosc`).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
okno = Window.partitionBy("klient_id").orderBy("data_zamowienia")

z_historia = zamowienia_z_wartoscia.withColumn(
    "wartosc_poprzedniego", F.lag("wartosc", 1).over(okno)
).withColumn(
    "wartosc_przedpoprzedniego", F.lag("wartosc", 2).over(okno)
)

rosnace = z_historia.filter(
    (F.col("wartosc_przedpoprzedniego") < F.col("wartosc_poprzedniego"))
    & (F.col("wartosc_poprzedniego") < F.col("wartosc"))
)
rosnace.select(
    "klient_id", "data_zamowienia", "wartosc_przedpoprzedniego",
    "wartosc_poprzedniego", "wartosc"
).orderBy("klient_id", "data_zamowienia").show(20)
```

Zauwaz, ze potrzebujemy tu wyniku z sekcji 3 (zamowienia_z_wartoscia) -- jesli uruchamiasz to cwiczenie osobno, najpierw policz wartosc kazdego zamowienia tak jak w sekcji 3.

</details>

## Co dalej?

W **Module 8** poznamy UDF-y (funkcje zdefiniowane przez użytkownika) -- kiedy są
potrzebne, jaki mają koszt wydajnościowy w porównaniu z funkcjami wbudowanymi, i czym są
pandas UDF-y.